# Getting Started, Chapter 3 -- Fitting a model to data

The SIR priors from Chapter 1 say what we believe *before* seeing data. This
chapter calibrates `beta` and `gamma` to a synthetic outbreak using
Approximate Bayesian Computation (ABC).

In [1]:
import numpy as np
from scipy.integrate import solve_ivp

from sims_pars.fit import DataModel, Particle, ApproxBayesCom, ApproxBayesComSMC
from sims_pars.fit.targets import read_targets

## 1. A forward simulator

A deterministic SIR ODE. Given `beta` and `gamma`, it returns the infected
**prevalence** (fraction of the population) on a handful of days.

In [2]:
DAYS = (2, 4, 6, 8, 10, 12, 14)
POP = 1000

def solve_sir(beta, gamma, i0=5, pop=POP):
    def rhs(t, y):
        s, i, r = y
        n = s + i + r
        return [-beta * s * i / n, beta * s * i / n - gamma * i, gamma * i]
    sol = solve_ivp(rhs, [0, max(DAYS)], [pop - i0, i0, 0], t_eval=DAYS, rtol=1e-8)
    return {f'prev_d{d}': i / pop for d, (s, i, r) in zip(DAYS, sol.y.T)}

truth = solve_sir(beta=1.8, gamma=0.6)
{k: round(v, 4) for k, v in truth.items()}

{'prev_d2': np.float64(0.0499),
 'prev_d4': np.float64(0.2464),
 'prev_d6': np.float64(0.2663),
 'prev_d8': np.float64(0.1321),
 'prev_d10': np.float64(0.0536),
 'prev_d12': np.float64(0.0206),
 'prev_d14': np.float64(0.0078)}

## 2. Wrap it in a `DataModel`

A `DataModel` ties three things together:

* the **PCore script** -- the priors and any derived parameters,
* the **targets** -- what was observed, via `read_targets`,
* a **`simulate`** method -- prior draw in, `Particle` (params + simulated
  outputs) out.

In [3]:
SIR_SCRIPT = '''
PCore SIR {
    beta  ~ unif(0.8, 3.0)
    gamma ~ unif(0.2, 1.2)
    r0 = beta / gamma
}
'''

class SIRModel(DataModel):
    def __init__(self, observed):
        DataModel.__init__(self, read_targets(observed, error=0.1), SIR_SCRIPT)

    def simulate(self, pars):
        sim = solve_sir(pars['beta'], pars['gamma'])
        return Particle(pars, sim)

model = SIRModel(truth)
print('free parameters:', model.FreeParameters)
model.Domain

free parameters: ['beta', 'gamma']


[Domain(beta, Double, LU=[0.8, 3.0], LS=[1.9000000000000001, 0.6350852961085883])),
 Domain(gamma, Double, LU=[0.2, 1.2], LS=[0.7, 0.28867513459481287]))]

In [4]:
# one prior draw, simulated and scored
p = model.sample_prior()
sim = model.simulate(p)
print('draw    :', dict(p))
print('distance:', round(model.calc_distance(sim), 3))

draw    : {'beta': np.float64(2.2886048799475294), 'gamma': np.float64(0.33179045953590264), 'r0': np.float64(6.897741674515756)}
distance: 37.497


## 3. Fit with ABC

`ApproxBayesCom` draws from the prior, keeps the draws whose simulated output
lands within a distance `eps` of the data, and returns those as the posterior.
`eps` is the `p_test` quantile of a batch of prior distances -- a single,
aggressive cut. It is quick, but a hard threshold on few surviving draws can
give a deceptively tight posterior (watch whether its spread actually covers
the truth below).

In [5]:
abc = ApproxBayesCom(verbose=0)
abc.fit(model)
post = abc.sample_posteriors(500)
post.to_df()[['beta', 'gamma', 'r0']].describe().round(3)

01-09-2026 18:16:51 INFO: Sample prior


01-09-2026 18:16:51 INFO: Start a parallel sampler for collecting test runs


01-09-2026 18:16:52 INFO: Prior_Yield:  100.00%


01-09-2026 18:16:52 INFO: Eps: 3.27553


,beta,gamma,r0
count,500.000,500.000,500.000
mean,1.887,0.606,3.138
std,0.153,0.064,0.318
min,1.638,0.498,2.739
25%,1.787,0.556,2.977
50%,1.891,0.598,3.059
75%,2.066,0.652,3.297
max,2.084,0.700,3.797


## 4. Fit with ABC-SMC

`ApproxBayesComSMC` walks a *decreasing* sequence of `eps` thresholds,
resampling and perturbing the particle population each round. It keeps a full
population at every step, so the posterior spread it reports is more
trustworthy than a single-shot cut -- here it stays wide enough to cover the
truth.

In [6]:
smc = ApproxBayesComSMC(n_iter=500, max_round=14, verbose=0)
smc.fit(model)
smc_post = smc.sample_posteriors(500)
smc_post.to_df()[['beta', 'gamma', 'r0']].describe().round(3)

01-09-2026 18:16:55 INFO: Initialising


01-09-2026 18:16:56 INFO: Round 0, ESS 500.00


01-09-2026 18:16:56 INFO: Round 1, ESS 448, Epsilon 43.6627, Acceptance 91.4%


01-09-2026 18:16:57 INFO: Round 2, ESS 409, Epsilon 28.5414, Acceptance 79.6%


01-09-2026 18:16:57 INFO: Round 3, ESS 372, Epsilon 24.4726, Acceptance 71.6%


...

01-09-2026 18:17:02 INFO: Round 12, ESS 500, Epsilon 8.6808, Acceptance 64.4%


01-09-2026 18:17:02 INFO: Round 13, ESS 450, Epsilon 8.0247, Acceptance 59.2%


01-09-2026 18:17:03 INFO: Round 14, ESS 403, Epsilon 7.3635, Acceptance 58.4%


01-09-2026 18:17:03 INFO: Collecting posterior


,beta,gamma,r0
count,500.000,500.000,500.000
mean,1.902,0.693,2.853
std,0.205,0.147,0.608
min,1.516,0.432,1.929
25%,1.736,0.575,2.327
50%,1.910,0.694,2.791
75%,2.085,0.792,3.272
max,2.285,0.979,4.374


The round-by-round schedule is on the monitor:

In [7]:
smc.Monitor.Trajectories

,Round,Eval,Eps,ESS,Acc
Time,,,,,
0,0,500,inf,500.0,1.000
1,1,500,43.662658,448.0,0.914
2,2,500,28.541372,409.0,0.796
3,3,500,24.472585,372.0,0.716
4,4,500,21.335023,337.0,0.688
5,5,500,18.125907,305.0,0.678
6,6,500,15.688758,500.0,0.740
7,7,500,13.611720,450.0,0.662
8,8,500,12.168431,405.0,0.612


Against the truth (`beta` 1.8, `gamma` 0.6, `R0` 3.0) -- the posterior
concentrates around it, with `gamma` (and hence `R0`) a little less tightly
pinned down by prevalence alone:

In [8]:
df = smc_post.to_df()
import pandas as pd
pd.DataFrame({
    'posterior mean': df[['beta', 'gamma', 'r0']].mean(),
    'posterior sd':   df[['beta', 'gamma', 'r0']].std(),
    'truth':          {'beta': 1.8, 'gamma': 0.6, 'r0': 3.0},
}).round(3)

,posterior mean,posterior sd,truth
beta,1.902,0.205,1.8
gamma,0.693,0.147,0.6
r0,2.853,0.608,3.0


## 5. Posterior-predictive check

Simulate from the posterior draws and compare to the observed prevalence.

In [9]:
pred = smc_post.to_pred_df()
pd.DataFrame({
    'observed':        truth,
    'predicted mean':  pred.mean(),
    'predicted 5%':    pred.quantile(0.05),
    'predicted 95%':   pred.quantile(0.95),
}).round(4)

,observed,predicted mean,predicted 5%,predicted 95%
prev_d2,0.0499,0.0539,0.0280,0.0940
prev_d4,0.2464,0.2339,0.1100,0.3853
prev_d6,0.2663,0.2202,0.1449,0.3203
prev_d8,0.1321,0.1101,0.0516,0.1863
prev_d10,0.0536,0.0454,0.0154,0.0879
prev_d12,0.0206,0.0177,0.0044,0.0390
prev_d14,0.0078,0.0068,0.0012,0.0167


## Other algorithms

The same `DataModel` plugs into the other fitters:

* **`GeneticAlg`** (`from sims_pars.fit import GeneticAlg`) -- a real-coded GA
  for a MAP / best-fit point estimate rather than a full posterior.
* **`BayesHistoryMatching`** (`from sims_pars.fit.hme import BayesHistoryMatching`)
  -- history matching with a Gaussian-process emulator; needs the optional
  extra, `pip install "sims-pars[hme]"`.

See the **Tutorials** section for a deeper treatment of each.

In [10]:
from sims_pars.fit import GeneticAlg

ga = GeneticAlg(n_collect=120, max_round=15, parallel=False, verbose=0)
ga.fit(model)
print('best fit:', {k: round(v, 3) for k, v in dict(ga.State.Best.Pars).items()})

01-09-2026 18:17:03 INFO: Initialising


Genesis:   0%|          | 0/120 [00:00<?, ?it/s]

Genesis:  32%|███▏      | 38/120 [00:00<00:00, 374.23it/s]

Genesis:  65%|██████▌   | 78/120 [00:00<00:00, 386.81it/s]

Genesis:  98%|█████████▊| 118/120 [00:00<00:00, 391.24it/s]

...

Evaluate:  34%|███▍      | 41/120 [00:00<00:00, 407.24it/s]

Evaluate:  68%|██████▊   | 82/120 [00:00<00:00, 401.38it/s]

Evaluate: 100%|██████████| 120/120 [00:00<00:00, 404.75it/s]


01-09-2026 18:17:09 INFO: Round 15, Max fitness -0.7893, Mean fitness -0.7902


best fit: {'beta': 1.801, 'gamma': 0.598, 'r0': 3.014}


---
That is the whole loop: **describe** a model as a PCore script, **sample** and
**intervene** on it, then **fit** it to data. From here, the Tutorials go deeper
into each piece and the API reference documents every class.